In [1]:
import os
import pandas as pd
import numpy as np


### Data Cleaning Notebook
This notebook is made to clean data retrieved from the database called DVI CEIF, that stores all properties sold in France between 2014 and today. Since you can't, on that website, extract a lot of data at once, the extraction was done arrondissement by arrondissement, for appartments sold between January 1st and June 30th of 2025 (most recent data available), of size between 30 and 120 square meters. The goal is to have ~200 - 500 references per arrondissement. 

The raw files were downloaded as .xlsx files, then converted to csv files and sorted raw in the "raw_data" directory.  

This notebook retrieves these csv files and cleans them, then stores them in the "data" directory. 

In [19]:
dataPath = os.path.join(os.getcwd(), "..", "raw_data", "DVICeif_data_Paris")
outputPath = os.path.join(os.getcwd(), "..", "data")

In [23]:
os.makedirs(os.path.join(outputPath, "per_arrondissement"), exist_ok=True)

### Clean CSV per arondissement

In [24]:
outputPath_per_arrondissement = os.path.join(outputPath, "per_arrondissement")

In [26]:
#FOR PARIS 1 TO 9
for i in range(1, 10): 
    filePath = f"{dataPath}/Mutations2025-DVICeif-7500{i}.csv"
    df = pd.read_csv(filePath, sep=";")

    df_new = pd.DataFrame({
        "date_mutation": pd.to_datetime(df["Date de vente"], format="%d/%m/%Y", errors="coerce").dt.strftime("%Y-%m-%d"),
        
        "surface_reelle_bati": pd.to_numeric(df["Surface Bâtie"], errors="coerce").astype(float),
        
        "nombre_pieces_principales": pd.to_numeric(df["Nombre de pièces"], errors="coerce").astype(float),
        
        "code_departement": pd.to_numeric(df["Département"], errors="coerce").astype("Int64"),
        
        "code_postal": pd.to_numeric(df["Code postal"], errors="coerce").astype("Int64"),
        
        "longitude": pd.to_numeric(
            df["Coordonnées X"].astype(str).str.replace(",", ".", regex=False),
            errors="coerce"
        ),
        
        "latitude": pd.to_numeric(
            df["Coordonnées Y"].astype(str).str.replace(",", ".", regex=False),
            errors="coerce"
        ),
        
        "type_local": (
            df["Nature des biens"]
            .astype(str)
            .str.strip()
            .str.replace(r"^Un\s+", "", regex=True)
            .str.replace(r"^Une\s+", "", regex=True)
            .str.capitalize()
        ),
        
        "valeur_fonciere": pd.to_numeric(
            df["Montant de la cession hors droits"].astype(str).str.replace(",", ".", regex=False),
            errors="coerce"
        ).astype(float)
    })

    # Sauvegarder au format demandé
    df_new.to_csv(os.path.join(outputPath_per_arrondissement, f"Mutations2025-7500{i}-Clean.csv"), sep=";", index=False)
    print(f"Processed file: Mutations2025-7500{i}.csv")

Processed file: Mutations2025-75001.csv
Processed file: Mutations2025-75002.csv
Processed file: Mutations2025-75003.csv
Processed file: Mutations2025-75004.csv
Processed file: Mutations2025-75005.csv
Processed file: Mutations2025-75006.csv
Processed file: Mutations2025-75007.csv
Processed file: Mutations2025-75008.csv
Processed file: Mutations2025-75009.csv


In [27]:
#FOR PARIS 10 TO 20
for i in range(10, 21): 
    filePath = f"{dataPath}/Mutations2025-DVICeif-750{i}.csv"
    df = pd.read_csv(filePath, sep=";")

    df_new = pd.DataFrame({
        "date_mutation": pd.to_datetime(df["Date de vente"], format="%d/%m/%Y", errors="coerce").dt.strftime("%Y-%m-%d"),
        
        "surface_reelle_bati": pd.to_numeric(df["Surface Bâtie"], errors="coerce").astype(float),
        
        "nombre_pieces_principales": pd.to_numeric(df["Nombre de pièces"], errors="coerce").astype(float),
        
        "code_departement": pd.to_numeric(df["Département"], errors="coerce").astype("Int64"),
        
        "code_postal": pd.to_numeric(df["Code postal"], errors="coerce").astype("Int64"),
        
        "longitude": pd.to_numeric(
            df["Coordonnées X"].astype(str).str.replace(",", ".", regex=False),
            errors="coerce"
        ),
        
        "latitude": pd.to_numeric(
            df["Coordonnées Y"].astype(str).str.replace(",", ".", regex=False),
            errors="coerce"
        ),
        
        "type_local": (
            df["Nature des biens"]
            .astype(str)
            .str.strip()
            .str.replace(r"^Un\s+", "", regex=True)
            .str.replace(r"^Une\s+", "", regex=True)
            .str.capitalize()
        ),
        
        "valeur_fonciere": pd.to_numeric(
            df["Montant de la cession hors droits"].astype(str).str.replace(",", ".", regex=False),
            errors="coerce"
        ).astype(float)
    })

    # Sauvegarder au format demandé
    df_new.to_csv(os.path.join(outputPath_per_arrondissement, f"Mutations2025-750{i}-Clean.csv"), sep=";", index=False)
    print(f"Processed file: Mutations2025-750{i}.csv")

Processed file: Mutations2025-75010.csv
Processed file: Mutations2025-75011.csv
Processed file: Mutations2025-75012.csv
Processed file: Mutations2025-75013.csv
Processed file: Mutations2025-75014.csv
Processed file: Mutations2025-75015.csv
Processed file: Mutations2025-75016.csv
Processed file: Mutations2025-75017.csv
Processed file: Mutations2025-75018.csv
Processed file: Mutations2025-75019.csv
Processed file: Mutations2025-75020.csv


### Combine everything in one big CSV

In [41]:
import pandas as pd
import glob

files = glob.glob(os.path.join(outputPath_per_arrondissement, "Mutations2025-750*-Clean.csv"))

df1 = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)
print(df1.columns)

#Sort by date, from most recent to oldest mutation 
df1 = df1.sort_values(by=df1.columns[0], ascending=False)  #first column is the date column 

df1.to_csv(os.path.join(outputPath, "paris_references_DVICeif.csv"), index=False)

Index(['date_mutation;surface_reelle_bati;nombre_pieces_principales;code_departement;code_postal;longitude;latitude;type_local;valeur_fonciere'], dtype='object')
